# support-agent-mcp: full evaluation on a free Colab T4 GPU

Runs the 60-task eval (45 normal + 15 prompt-injection tasks) with **qwen2.5:7b** served by Ollama on the GPU, for prompt **v1** (baseline) and **v2** (KB router + grounding rules). Free; no paid API.

Runtime > Change runtime type > **T4 GPU**, then Runtime > Run all. Downloads two JSON result files at the end.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
import subprocess, time
subprocess.Popen('nohup ollama serve > /content/ollama.log 2>&1 &', shell=True)
time.sleep(5)
!ollama pull qwen2.5:7b > /dev/null && ollama pull nomic-embed-text > /dev/null && ollama list

In [ ]:
!git clone -q https://github.com/ryannaidji/support-agent-mcp.git
%cd support-agent-mcp
%pip install -q -r requirements.txt
!python -m pytest -q 2>&1 | tail -1

In [ ]:
!PYTHONUNBUFFERED=1 python -m evals.run_eval --model qwen2.5:7b --graph react --prompt v1 --out evals/results/qwen2.5-7b_react_v1_t4.json 2>&1 | grep -E '^\[|^\{|^  "|^\}'

In [ ]:
!PYTHONUNBUFFERED=1 python -m evals.run_eval --model qwen2.5:7b --graph react --prompt v2 --out evals/results/qwen2.5-7b_react_v2_t4.json 2>&1 | grep -E '^\[|^\{|^  "|^\}'

In [ ]:
from google.colab import files
files.download('evals/results/qwen2.5-7b_react_v1_t4.json')
files.download('evals/results/qwen2.5-7b_react_v2_t4.json')

Put both JSON files in `support-agent-mcp/evals/results/` on your machine and tell Claude.